In [ ]:
# %pip install sentence-transformers
# %pip install pinecone

In [ ]:
import pandas as pd, re, os
from pypdf import PdfReader
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

In [ ]:
df_tkt = pd.read_csv("ticket_types.csv")
reader = PdfReader("tickets.pdf")

data = ''

for page in reader.pages:
    data+= page.extract_text()

In [ ]:
probdesc = []

# Regex breakdown:
# <(\d+)>   : Matches the opening tag and captures the number inside (e.g., 1 or 2)
# (.*?)     : Non-greedy match that captures all the text inside the tags
# </\1>     : Matches the closing tag, ensuring the number matches the opening tag (\1)
pattern = r"<(\d+)>(.*?)</\1>"

matches = re.findall(pattern, data, re.DOTALL)

# Print the results
for tag_num, content in matches:
    # print(f"--- Tag {tag_num} ---")
    probdesc.append(content.strip())
    # print()

In [ ]:
key = os.environ["PINECONE_KEY"]
embedding_model = SentenceTransformer('all-mpnet-base-v2')

In [50]:
embedding = embedding_model.encode("learning new technology")
emb_shape = embedding.shape

In [ ]:
# initialize pinecone with the API Key
pc = Pinecone(api_key=key)

In [53]:
# Now do stuff
pc.list_indexes().names()

[]

In [ ]:
# pc.delete_index(ndx)

In [56]:
ndx = "tickets"

if ndx not in pc.list_indexes().names():
        pc.create_index( name=ndx, dimension=emb_shape[0], metric='euclidean',
                        spec=ServerlessSpec(cloud='aws', region='us-east-1') )

In [ ]:
knowledge_base = []

for i, rec in df_tkt.iterrows():
    
    severity = rec["severity"]
    probtype = rec["problem_type"]
    probdate = rec["problem_date"]

    desc = probdesc[i]
    desc = ' '.join(desc.split())

    row = { "id": str(i), 
            "values": embedding_model.encode(desc).tolist(),
            "metadata": {
                'priority':rec["priority"], 
                'severity':rec["severity"],
                'problemtype':rec["problem_type"],
                'problemdate':rec["problem_date"]
                        }
          }
    
    knowledge_base.append(row)

In [ ]:
index_info = pc.describe_index(ndx)
index = pc.Index(host=index_info.host)

# fetch the statistics
stats = index.describe_index_stats()

# get the namespaces
namespace = stats.get('namespaces',{})
print(f"Existing namespaces\n{namespace}")

In [57]:
# Add the data into the VDB
index.upsert(vectors=knowledge_base)

UpsertResponse(upserted_count=100)

In [ ]:
# fetch the data

index.fetch(ids=["1"])

FetchResponse(vectors={'1': Vector(id='1', values=[0.00408477243, -0.00688387174, -0.0258448515, ...765 more], sparse_values=None, metadata={'priority': 'Critical', 'problemdate': '21-08-2024', 'problemtype': 'Disk Full', 'severity': 'S3'})}, namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Wed, 17 Jun 2026 16:04:41 GMT', 'content-type': 'application/json', 'content-length': '10538', 'connection': 'keep-alive', 'x-pinecone-request-latency-ms': '161', 'x-envoy-upstream-service-time': '162', 'x-pinecone-response-duration-ms': '170', 'grpc-status': '0', 'server': 'envoy'}))

In [ ]:
query = "What are the high-severity network connection ticket issues?"
query_vector = embedding_model.encode(query).tolist()

In [73]:
query_response = index.query(vector=query_vector, top_k=3, include_metadata=True)
print(query_response)

QueryResponse(matches=[ScoredVector(id='52', score=1.1900624, values=[], metadata={'priority': 'Critical', 'problemdate': '26-02-2024', 'problemtype': 'Application Crash', 'severity': 'S2'}), ScoredVector(id='50', score=1.1900624, values=[], metadata={'priority': 'Low', 'problemdate': '19-02-2024', 'problemtype': 'Compliance Violation', 'severity': 'S1'}), ScoredVector(id='56', score=1.27531242, values=[], metadata={'priority': 'Critical', 'problemdate': '01-12-2023', 'problemtype': 'Data Corruption', 'severity': 'S2'})], namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Wed, 17 Jun 2026 16:13:16 GMT', 'content-type': 'application/json', 'content-length': '524', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '1', 'x-pinecone-request-latency-ms': '364', 'x-envoy-upstream-service-time': '139', 'x-pinecone-response-duration-ms': '365', 'grpc-status': '0', 'server': 'envoy'}))


In [85]:
print(f"Query : -> {query}")
print("Closest matches\n")

# for match in query_response.matches:
for i in range(len(query_response.matches)):
    match = query_response.matches[i]
    cid = int(match.id)
    md = match.metadata

    print(f"Match : {i+1}")
    print(probdesc[cid])
    print() 

Query : -> What are the high-severity network connection ticket issues?
Closest matches

Match : 1
A critical network outage impacted the primary data center interconnecting regional banking 
branches across the southern zone. The failure originated from a misconfigured BGP route update 
that propagated incorrect routing tables to edge routers, effectively isolating three branch offices 
from the central transaction hub. ATM networks and point-of-sale terminals in affected branches 
went offline, disrupting cash withdrawal and payment services for approximately ninety minutes. 
Network engineers coordinated with the ISP to revert the faulty configuration while activating backup MPLS links to restore partial connectivity. A post-incident review was scheduled to evaluate 
network change management controls.

Match : 2
A critical network outage impacted the primary data center interconnecting regional banking 
branches across the southern zone. The failure originated from a misconfigured 

In [98]:
# Metadata Search
query_response = index.query( vector=query_vector, top_k=3, include_metadata=True,
                            filter={ "severity": {"$eq": "S2"} }
                            # "problemtype": {"$in": ["Network", "Cloud"]}  
                            )

In [100]:
# 3. Print your cleanly filtered results
for match in query_response['matches']:
    cid = int(match['id'])
    print(f"ID: {match['id']}, MetaData: {match['metadata']}")
    print(probdesc[cid][:50])
    print("*" * 50)

ID: 52, MetaData: {'priority': 'Critical', 'problemdate': '26-02-2024', 'problemtype': 'Application Crash', 'severity': 'S2'}
A critical network outage impacted the primary dat
**************************************************
ID: 26, MetaData: {'priority': 'Critical', 'problemdate': '20-05-2023', 'problemtype': 'Authentication Failure', 'severity': 'S2'}
An unplanned hardware failure in the primary SAN s
**************************************************
ID: 56, MetaData: {'priority': 'Critical', 'problemdate': '01-12-2023', 'problemtype': 'Data Corruption', 'severity': 'S2'}
The cash management portal used by corporate treas
**************************************************
